Grupo 02 - D2 NYC Taxi

# Bloco 01 | Escolha do dataset

**Dataset escolhido:** D2 - NYC Táxi amarelo, janeiro de 2024, mais a tabela de zonas.

Fontes:
- `yellow_tripdata_2024-01.parquet` (viagens)
- `taxi_zone_lookup.csv` (zonas)

**P1. Qual dataset o grupo escolheu e por que ele foi preferido ao segundo candidato?**

Escolhemos o D2 (NYC Táxi). Ele atende ao critério de volume e qualidade: uma fonte com 2.964.624 linhas (muito mais que as 10 mil exigidas) e uma segunda fonte (zonas) que precisa ser combinada. Os dois arquivos baixam sem cadastro, e a coluna de data de embarque permite agregar por hora.
Preferimos o D2 (NYC Táxi) ao D4 (Banco Central: Selic, IPCA e dólar) porque o volume de dados é consideravelmente maior, deixando o tratamento e a análise mais interessantes.

**P2. Quantas fontes ele possui e quantas linhas tem a maior delas?**

O dataset tem **2 fontes**: o arquivo de viagens (parquet) e o arquivo de zonas (CSV). A maior é a de viagens, com **2.964.624 linhas** e 19 colunas. A de zonas tem 265 linhas e 4 colunas.

**P3. Que problema de qualidade vocês já enxergaram apenas olhando uma amostra?**

Olhando as amostras e rodando contagens simples, encontramos vários problemas na tabela de viagens (2.964.624 linhas) e um na de zonas:

1. **Nulos em `passenger_count`:** 140.162 viagens (cerca de 4,7%) não têm a quantidade de passageiros. Isso também explica por que a coluna vem como `float64` em vez de inteiro.
2. **Valores monetários inválidos:** 35.504 viagens (cerca de 1,2%) têm `total_amount` negativo e 416 têm `total_amount` igual a zero. Uma receita negativa não faz sentido para uma corrida normal e distorceria a soma por zona e hora.
3. **Distância zero:** 60.371 viagens (cerca de 2,0%) têm `trip_distance` igual a zero, e 31.465 têm `passenger_count` igual a zero. São registros suspeitos, e ainda não sabemos se são corridas canceladas ou erros de registro.
4. **Datas fora do período:** 18 embarques estão fora de janeiro de 2024, em um arquivo que deveria conter só esse mês.
5. **Nulos na tabela de zonas:** `Borough` tem 1 nulo, `Zone` tem 1 e `service_zone` tem 2, em 265 linhas.

# BLOCO 2 | Conhecer a fonte e definir a pergunta

In [5]:
import os
from pathlib import Path

print("Trabalhando em:", os.getcwd())
print("Pasta 'fonte' existe?", Path("fonte").exists())

Trabalhando em: c:\desenvolvimento-faculdade\I.A-C.D-Grupo-2-2-periodo--Faculdade-Donaduzzi\pipeline_de_dados
Pasta 'fonte' existe? True


In [9]:
import pandas as pd
from pathlib import Path

PASTA_FONTE = Path("fonte")

# Fonte 1: viagens
df_viagens = pd.read_parquet(PASTA_FONTE / 'yellow_tripdata_2024-01.parquet')

print("shape:", df_viagens.shape)
print()
display(df_viagens.head())
print()
df_viagens.info(show_counts=True)

shape: (2964624, 19)



,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0



<class 'pandas.DataFrame'>
RangeIndex: 2964624 entries, 0 to 2964623
Data columns (total 19 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   VendorID               2964624 non-null  int32         
 1   tpep_pickup_datetime   2964624 non-null  datetime64[us]
 2   tpep_dropoff_datetime  2964624 non-null  datetime64[us]
 3   passenger_count        2824462 non-null  float64       
 4   trip_distance          2964624 non-null  float64       
 5   RatecodeID             2824462 non-null  float64       
 6   store_and_fwd_flag     2824462 non-null  str           
 7   PULocationID           2964624 non-null  int32         
 8   DOLocationID           2964624 non-null  int32         
 9   payment_type           2964624 non-null  int64         
 10  fare_amount            2964624 non-null  float64       
 11  extra                  2964624 non-null  float64       
 12  mta_tax                2964624 non-nul

In [10]:
# Contagem de nulos por coluna (viagens)
nulos_viagens = df_viagens.isna().sum()
display(nulos_viagens.to_frame("nulos").assign(pct=(nulos_viagens / len(df_viagens) * 100).round(2)))

,nulos,pct
VendorID,0,0.00
tpep_pickup_datetime,0,0.00
tpep_dropoff_datetime,0,0.00
passenger_count,140162,4.73
trip_distance,0,0.00
RatecodeID,140162,4.73
store_and_fwd_flag,140162,4.73
PULocationID,0,0.00
DOLocationID,0,0.00
payment_type,0,0.00


In [11]:
# Fonte 2: zonas
df_zonas = pd.read_csv(PASTA_FONTE /'taxi_zone_lookup.csv')

print("shape:", df_zonas.shape)
print()
display(df_zonas.head())
print()
df_zonas.info()
print()
print("Nulos por coluna:")
display(df_zonas.isna().sum())

shape: (265, 4)



,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone



<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   LocationID    265 non-null    int64
 1   Borough       264 non-null    str  
 2   Zone          264 non-null    str  
 3   service_zone  263 non-null    str  
dtypes: int64(1), str(3)
memory usage: 17.0 KB

Nulos por coluna:


LocationID      0
Borough         1
Zone            1
service_zone    2
dtype: int64

In [12]:
# Os nulos das zonas são nulos de verdade? Olhem as linhas com nulo.
# Dica: o pandas converte os textos "NA" e "N/A" em NaN automaticamente.
display(df_zonas[df_zonas.isna().any(axis=1)])

# Compare que a leitura que NÃO converte texto em nulo:
df_zonas_raw = pd.read_csv(PASTA_FONTE / 'taxi_zone_lookup.csv', keep_default_na=False)
display(df_zonas_raw[df_zonas_raw['LocationID'].isin(df_zonas[df_zonas.isna().any(axis=1)]['LocationID'])])

,LocationID,Borough,Zone,service_zone
263,264,Unknown,NaN,NaN
264,265,NaN,Outside of NYC,NaN


,LocationID,Borough,Zone,service_zone
263,264,Unknown,N/A,N/A
264,265,N/A,Outside of NYC,N/A


In [13]:
# Chave de ligação e coluna de data
print("Chave: viagens.PULocationID  ->  zonas.LocationID")
print("IDs de embarque que não existem na tabela de zonas:",
      (~df_viagens['PULocationID'].isin(df_zonas['LocationID'])).sum())

# Coluna de data usada para agregar por hora
print()
print("Primeiro embarque:", df_viagens['tpep_pickup_datetime'].min())
print("Último embarque:  ", df_viagens['tpep_pickup_datetime'].max())

Chave: viagens.PULocationID  ->  zonas.LocationID
IDs de embarque que não existem na tabela de zonas: 0

Primeiro embarque: 2002-12-31 22:59:39
Último embarque:   2024-02-01 00:01:15


In [14]:
# Problemas de qualidade visíveis (usar esses números na P3 do Bloco 1)
inicio, fim = pd.Timestamp('2024-01-01'), pd.Timestamp('2024-02-01')
fora_periodo = ~df_viagens['tpep_pickup_datetime'].between(inicio, fim, inclusive='left')

print("Embarques fora de jan/2024:    ", fora_periodo.sum())
print("total_amount negativo:         ", (df_viagens['total_amount'] < 0).sum())
print("total_amount igual a zero:     ", (df_viagens['total_amount'] == 0).sum())
print("trip_distance igual a zero:    ", (df_viagens['trip_distance'] == 0).sum())
print("passenger_count nulo:          ", df_viagens['passenger_count'].isna().sum())
print("passenger_count igual a zero:  ", (df_viagens['passenger_count'] == 0).sum())

Embarques fora de jan/2024:     18
total_amount negativo:          35504
total_amount igual a zero:      416
trip_distance igual a zero:     60371
passenger_count nulo:           140162
passenger_count igual a zero:   31465


### Respostas do Bloco 2

**P1. Que tabela agregada responde à pergunta do grupo? Nessa tabela, uma linha representa o quê?**

A tabela é a **receita por zona de embarque e hora do dia** (`receita_zona_hora`). Cada linha representa **uma zona de embarque em uma hora do dia** (por exemplo, a zona X às 18h), com a receita total e, pelo menos, uma segunda métrica (número de corridas, por exemplo).

**P2. Quais fontes ou colunas NÃO são necessárias para a pergunta? Por que deixá-las de fora é uma decisão da etapa de Extract?**

Das 19 colunas de viagens, a pergunta só precisa de: data/hora de embarque (`tpep_pickup_datetime`), zona de embarque (`PULocationID`) e valores (`fare_amount`, `total_amount`). Mantivemos também `trip_distance` para uma possível segunda pergunta (Bloco 9).
Ficam de fora, por exemplo: `tpep_dropoff_datetime`, `DOLocationID`, `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `payment_type`, e as taxas e sobretaxas.
Deixar de fora é decisão do Extract porque é nessa etapa que escolhemos o que entra no pipeline: ler só as colunas necessárias reduz memória, tempo e tamanho da Bronze, e o que não entra na Bronze não precisa ser tratado nas camadas seguintes.

**P3. Algum nulo encontrado pode estar carregando informação em vez de indicar erro? Se sim, qual é o nulo e o que ele significa?**

Sim. Na tabela de zonas, os nulos de `Borough`, `Zone` e `service_zone` não são falta de dado: são textos que o pandas converteu em nulo (`NA`, `N/A`). Eles aparecem nas linhas de `LocationID` 264 e 265, que representam corridas com zona **desconhecida** ou fora da cidade. Ou seja, o nulo significa "zona não identificada", e isso é uma informação.
_[Confiram na célula acima quais linhas apareceram e ajustem o texto com os valores exatos.]_

# BLOCO 3 | Extract: construir a Bronze
Regras da Bronze: **cópia fiel da fonte**, só com as colunas necessárias e com metadados de controle. Nenhuma correção.

In [15]:
from pathlib import Path
from datetime import datetime
import urllib.request

# Configuração (no Bloco 10 vocês vão ampliar este dicionário)
CONFIG = {
    "bronze": "data/bronze",
    "silver": "data/silver",
    "gold":   "data/gold",
    "fontes": {
        "viagens": {
            "arquivo_local": "fonte/yellow_tripdata_2024-01.parquet",
            "url": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
            "colunas": ["tpep_pickup_datetime", "PULocationID",
                        "fare_amount", "total_amount", "trip_distance"],
        },
        "zonas": {
            "arquivo_local": "fonte/taxi_zone_lookup.csv",
            "url": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
        },
    },
}

# Passo 1: criar as pastas das três camadas
for camada in ("bronze", "silver", "gold"):
    Path(CONFIG[camada]).mkdir(parents=True, exist_ok=True)
print("Pastas criadas:", [CONFIG[c] for c in ("bronze", "silver", "gold")])

Pastas criadas: ['data/bronze', 'data/silver', 'data/gold']


In [16]:
def _obter_arquivo(fonte):
    """Usa o arquivo da pasta 'fonte'. Se não achar, para com uma mensagem clara."""
    caminho = Path(fonte["arquivo_local"])
    if not caminho.exists():
        raise FileNotFoundError(f"Não achei {caminho}. Confira se a pasta 'fonte' veio do Git.")
    return caminho


def extract(destino_bronze):
    """Lê cada fonte necessária e grava na Bronze SEM corrigir nada.
    Acrescenta metadados de controle: _data_extracao e _origem."""
    destino = Path(destino_bronze)
    destino.mkdir(parents=True, exist_ok=True)
    agora = datetime.now().isoformat(timespec="seconds")

    # Fonte em arquivo (parquet): ler só as colunas necessárias
    f = CONFIG["fontes"]["viagens"]
    viagens = pd.read_parquet(_obter_arquivo(f), columns=f["colunas"])
    viagens["_data_extracao"] = agora
    viagens["_origem"] = f["url"]
    viagens.to_parquet(destino / "viagens.parquet", index=False)   # sobrescreve: não duplica

    # Fonte em arquivo (csv): keep_default_na=False preserva "NA" e "N/A" como texto, sem converter em nulo
    f = CONFIG["fontes"]["zonas"]
    zonas = pd.read_csv(_obter_arquivo(f), keep_default_na=False)
    zonas["_data_extracao"] = agora
    zonas["_origem"] = f["url"]
    zonas.to_parquet(destino / "zonas.parquet", index=False)

    return {"viagens": len(viagens), "zonas": len(zonas)}

In [17]:
def linhas_bronze():
    pasta = Path(CONFIG["bronze"])
    return {arq.stem: pd.read_parquet(arq).shape for arq in sorted(pasta.glob("*.parquet"))}

# Executar extract() duas vezes seguidas e anotar as linhas após cada execução
extract(CONFIG["bronze"])
print("Após a 1ª execução:", linhas_bronze())

extract(CONFIG["bronze"])
print("Após a 2ª execução:", linhas_bronze())

Após a 1ª execução: {'viagens': (2964624, 7), 'zonas': (265, 6)}
Após a 2ª execução: {'viagens': (2964624, 7), 'zonas': (265, 6)}


In [18]:
# Conferir a Bronze com a fonte original
bronze_viagens = pd.read_parquet(Path(CONFIG["bronze"]) / "viagens.parquet")
bronze_zonas = pd.read_parquet(Path(CONFIG["bronze"]) / "zonas.parquet")

print("Viagens - fonte:", df_viagens.shape[0], "linhas | Bronze:", bronze_viagens.shape)
print("Zonas   - fonte:", df_zonas.shape[0], "linhas | Bronze:", bronze_zonas.shape)
print()
display(bronze_viagens.head(3))
display(bronze_zonas.tail(3))

Viagens - fonte: 2964624 linhas | Bronze: (2964624, 7)
Zonas   - fonte: 265 linhas | Bronze: (265, 6)



,tpep_pickup_datetime,PULocationID,fare_amount,total_amount,trip_distance,_data_extracao,_origem
0,2024-01-01 00:57:55,186,17.7,22.70,1.72,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/trip-dat...
1,2024-01-01 00:03:00,140,10.0,18.75,1.80,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/trip-dat...
2,2024-01-01 00:17:06,236,23.3,31.30,4.70,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/trip-dat...


,LocationID,Borough,Zone,service_zone,_data_extracao,_origem
262,263,Manhattan,Yorkville West,Yellow Zone,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/misc/tax...
263,264,Unknown,N/A,N/A,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/misc/tax...
264,265,N/A,Outside of NYC,N/A,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/misc/tax...


### Respostas do Bloco 3

**P1. Quantas linhas e colunas tem cada arquivo da Bronze? Os números batem com os da fonte original?**

- `viagens.parquet`: **2.964.624 linhas** e **7 colunas** (as 5 colunas escolhidas + `_data_extracao` + `_origem`). A fonte original tem 2.964.624 linhas e 19 colunas.
- `zonas.parquet`: **265 linhas** e **6 colunas** (as 4 originais + os 2 metadados). A fonte tem 265 linhas e 4 colunas.

As linhas batem com a fonte nos dois arquivos. As colunas diferem de propósito: em viagens deixamos de fora 14 colunas (decisão do Extract) e nos dois arquivos acrescentamos os 2 metadados.
_[Confiram os números de colunas na saída da célula de conferência.]_

**P2. O que aconteceu com o número de linhas na segunda execução? Se mudou, por que mudou e como corrigir?**

O número de linhas **não mudou** (2.964.624 em viagens e 265 em zonas nas duas execuções), porque `extract()` grava cada arquivo com `to_parquet`, que **sobrescreve** o arquivo. Se tivéssemos acrescentado (append) em vez de sobrescrever, a segunda execução dobraria as linhas (5.929.248 em viagens). A correção seria sobrescrever a Bronze a cada execução, ou controlar duplicidade por uma chave.

**P3. Por que a Bronze não deve receber nenhuma correção, mesmo que o grupo já conheça os problemas do dado?**

Porque a Bronze é o registro fiel do que a fonte entregou. Se corrigirmos ali, perdemos o dado original: não conseguimos auditar o que foi alterado, nem refazer a limpeza caso uma regra mude. Com a Bronze intacta, qualquer correção fica na Silver, onde pode ser mudada e reprocessada sem baixar a fonte de novo. Por isso lemos as zonas com `keep_default_na=False`: assim `NA` e `N/A` continuam como texto, exatamente como vieram.

**P4. Se a fonte sair do ar amanhã, o que do pipeline continua funcionando?**

Tudo o que parte da Bronze: `transform()` lê só da Bronze e `load()` lê só da Silver, então Silver e Gold podem ser reconstruídas normalmente. Só o `extract()` deixa de funcionar (e a carga de dados novos), porque ele depende da fonte.

# Bloco 04 | Transform: construir a Silver

In [19]:
import json

# Parâmetros das regras de limpeza (no Bloco 10 eles vão para o dicionário CONFIG principal)
CONFIG["silver_arquivo"] = "viagens_silver.parquet"
CONFIG["regras"] = {
    "periodo_inicio": "2024-01-01",   # escopo: só janeiro de 2024
    "periodo_fim":    "2024-02-01",   # limite superior (exclusivo)
    "total_max":      1000,           # corrida acima disso é tratada como valor inválido (ver decisão no texto)
}


def transform(origem_bronze, destino_silver):
    """Lê SOMENTE a Bronze, padroniza, limpa, junta viagens + zonas e filtra o escopo.
    Grava a Silver e devolve um dicionário com as contagens de cada decisão."""
    origem, destino = Path(origem_bronze), Path(destino_silver)
    for arq in ("viagens.parquet", "zonas.parquet"):
        if not (origem / arq).exists():
            raise FileNotFoundError(f"Bronze não encontrada ({origem / arq}). Rode extract() antes de transform().")
    destino.mkdir(parents=True, exist_ok=True)

    R = CONFIG["regras"]
    ini, fim = pd.Timestamp(R["periodo_inicio"]), pd.Timestamp(R["periodo_fim"])
    log = {"regras": []}

    v = pd.read_parquet(origem / "viagens.parquet")
    z = pd.read_parquet(origem / "zonas.parquet")
    log["linhas_bronze_viagens"] = len(v)
    log["soma_total_bronze"] = round(float(v["total_amount"].sum()), 2)

    # 1) Padronizar nomes e tipos -------------------------------------------------
    v = v.rename(columns={
        "tpep_pickup_datetime": "data_embarque", "PULocationID": "id_zona_embarque",
        "fare_amount": "valor_tarifa", "total_amount": "valor_total", "trip_distance": "distancia_milhas"})
    z = z.rename(columns={"LocationID": "id_zona_embarque", "Borough": "bairro",
                          "Zone": "zona", "service_zone": "zona_servico"})
    v["data_embarque"] = pd.to_datetime(v["data_embarque"])
    for col in ("valor_tarifa", "valor_total", "distancia_milhas"):
        v[col] = pd.to_numeric(v[col], errors="coerce")
    v["id_zona_embarque"] = v["id_zona_embarque"].astype("int32")
    z["id_zona_embarque"] = z["id_zona_embarque"].astype("int32")
    log["nulos_criados_na_conversao"] = int(v[["valor_tarifa", "valor_total", "distancia_milhas"]].isna().sum().sum())

    # 2) Duplicados (os metadados de extração ficam fora da comparação) --------------
    colunas_dado = ["data_embarque", "id_zona_embarque", "valor_tarifa", "valor_total", "distancia_milhas"]
    antes = len(v)
    v = v.drop_duplicates(subset=colunas_dado, keep="first")
    log["duplicados_removidos"] = antes - len(v)
    log["linhas_apos_duplicados"] = len(v)

    # 3) Nulos, coluna a coluna ------------------------------------------------------
    log["nulos_viagens"] = {c: int(v[c].isna().sum()) for c in colunas_dado}
    antes = len(v)
    v = v.dropna(subset=colunas_dado)       # rede de segurança: remove só se houver nulo nas colunas usadas
    log["nulos_removidos"] = antes - len(v)
    # Zonas: "N/A" é texto que significa "sem informação" (zona 264 e 265) -> vira rótulo explícito
    rotulos = {"bairro": "Fora de NYC", "zona": "Zona desconhecida", "zona_servico": "Sem classificação"}
    log["zonas_rotuladas"] = {c: int((z[c] == "N/A").sum()) for c in rotulos}
    for c, r in rotulos.items():
        z[c] = z[c].replace({"N/A": r})

    # 4) Valores inválidos: cada regra conta o que afetou (isolado) e o que removeu (em sequência)
    regras = [
        ("data impossível (antes do período)", lambda d: d["data_embarque"] < ini),
        ("total_amount negativo",              lambda d: d["valor_total"] < 0),
        ("total_amount igual a zero",          lambda d: d["valor_total"] == 0),
        ("tarifa negativa",                    lambda d: d["valor_tarifa"] < 0),
        ("distância negativa",                 lambda d: d["distancia_milhas"] < 0),
        ("total_amount acima do limite",       lambda d: d["valor_total"] > R["total_max"]),
    ]
    for nome, f in regras:
        isolado = int(f(v).sum())
        m = f(v)
        log["regras"].append({"regra": nome, "acao": "removida", "afetados_isolado": isolado,
                              "removidos_em_sequencia": int(m.sum()),
                              "soma_total_removida": round(float(v.loc[m, "valor_total"].sum()), 2)})
        v = v[~m]
    # distância zero: contada, mas mantida (não afeta receita)
    m0 = v["distancia_milhas"] == 0
    log["regras"].append({"regra": "distância igual a zero", "acao": "mantida",
                          "afetados_isolado": int(m0.sum()), "removidos_em_sequencia": 0, "soma_total_removida": 0.0})

    # 5) Junção viagens + zonas ---------------------------------------------------------
    soma_antes, linhas_antes = float(v["valor_total"].sum()), len(v)
    v = v.merge(z[["id_zona_embarque", "bairro", "zona", "zona_servico"]],
                on="id_zona_embarque", how="left", validate="many_to_one")
    log["join"] = {"linhas_antes": linhas_antes, "linhas_depois": len(v),
                   "soma_antes": round(soma_antes, 2), "soma_depois": round(float(v["valor_total"].sum()), 2),
                   "sem_zona_correspondente": int(v["zona"].isna().sum())}

    # 6) Filtro de escopo da pergunta: só janeiro de 2024 -------------------------------
    m = v["data_embarque"] >= fim
    log["regras"].append({"regra": "escopo: embarque fora de jan/2024 (depois do período)", "acao": "removida",
                          "afetados_isolado": int(m.sum()), "removidos_em_sequencia": int(m.sum()),
                          "soma_total_removida": round(float(v.loc[m, "valor_total"].sum()), 2)})
    v = v[~m]

    # 7) Coluna derivada e gravação -----------------------------------------------------
    v["hora_embarque"] = v["data_embarque"].dt.hour.astype("int8")
    v = v.reset_index(drop=True)
    v.to_parquet(destino / CONFIG["silver_arquivo"], index=False)
    pd.DataFrame(log["regras"]).to_csv(destino / "_regras_transform.csv", index=False)

    log["linhas_silver"] = len(v)
    log["soma_total_silver"] = round(float(v["valor_total"].sum()), 2)
    return log


In [20]:
log_transform = transform(CONFIG["bronze"], CONFIG["silver"])

print("Linhas na Bronze (viagens):      ", f'{log_transform["linhas_bronze_viagens"]:,}')
print("Duplicados removidos:            ", f'{log_transform["duplicados_removidos"]:,}')
print("Nulos criados na conversão:      ", log_transform["nulos_criados_na_conversao"])
print("Nulos por coluna (viagens):      ", log_transform["nulos_viagens"])
print("Zonas com 'N/A' rotuladas:       ", log_transform["zonas_rotuladas"])
print()
tabela_regras = pd.DataFrame(log_transform["regras"])
display(tabela_regras)
print()
print("JOIN:", log_transform["join"])
print()
print("Linhas na Silver:", f'{log_transform["linhas_silver"]:,}')


Linhas na Bronze (viagens):       2,964,624
Duplicados removidos:             19
Nulos criados na conversão:       0
Nulos por coluna (viagens):       {'data_embarque': 0, 'id_zona_embarque': 0, 'valor_tarifa': 0, 'valor_total': 0, 'distancia_milhas': 0}
Zonas com 'N/A' rotuladas:        {'bairro': 1, 'zona': 1, 'zona_servico': 2}



,regra,acao,afetados_isolado,removidos_em_sequencia,soma_total_removida
0,data impossível (antes do período),removida,15,15,352.31
1,total_amount negativo,removida,35500,35500,-893347.54
2,total_amount igual a zero,removida,413,413,0.00
3,tarifa negativa,removida,2064,2064,5260.95
4,distância negativa,removida,0,0,0.00
5,total_amount acima do limite,removida,7,7,21342.80
6,distância igual a zero,mantida,56649,0,0.00
7,escopo: embarque fora de jan/2024 (depois do p...,removida,3,3,90.47



JOIN: {'linhas_antes': 2926606, 'linhas_depois': 2926606, 'soma_antes': 80322586.89, 'soma_depois': 80322586.89, 'sem_zona_correspondente': 0}

Linhas na Silver: 2,926,603


In [21]:
# Conferência da Silver e reconciliação linha a linha (P1)
silver = pd.read_parquet(Path(CONFIG["silver"]) / CONFIG["silver_arquivo"])
lb = log_transform["linhas_bronze_viagens"]
removidos_regras = sum(r["removidos_em_sequencia"] for r in log_transform["regras"])
reconciliacao = lb - log_transform["duplicados_removidos"] - log_transform["nulos_removidos"] - removidos_regras

print(f"Bronze {lb:,} - duplicados {log_transform['duplicados_removidos']:,} - nulos {log_transform['nulos_removidos']:,} "
      f"- regras {removidos_regras:,} = {reconciliacao:,}")
print(f"Linhas da Silver:  {len(silver):,}  ->  bate? {reconciliacao == len(silver)}")
print()
print("Soma total_amount na Bronze:", f'{log_transform["soma_total_bronze"]:,.2f}')
print("Soma total_amount na Silver:", f'{log_transform["soma_total_silver"]:,.2f}')
print()

# Impacto de cada regra sobre a receita (P3): quanto da receita da Bronze cada regra removeu
impacto = tabela_regras[tabela_regras["acao"] == "removida"].copy()
impacto["pct_da_receita_bronze"] = (impacto["soma_total_removida"] / log_transform["soma_total_bronze"] * 100).round(3)
impacto["pct_das_linhas_bronze"] = (impacto["removidos_em_sequencia"] / lb * 100).round(3)
display(impacto[["regra", "removidos_em_sequencia", "pct_das_linhas_bronze", "soma_total_removida", "pct_da_receita_bronze"]])

print()
silver.info()
display(silver.head())
print("Nulos por coluna na Silver:")
display(silver.isna().sum().to_frame("nulos").T)
print("Período da Silver:", silver["data_embarque"].min(), "->", silver["data_embarque"].max())


Bronze 2,964,624 - duplicados 19 - nulos 0 - regras 38,002 = 2,926,603
Linhas da Silver:  2,926,603  ->  bate? True

Soma total_amount na Bronze: 79,456,384.28
Soma total_amount na Silver: 80,322,496.42



,regra,removidos_em_sequencia,pct_das_linhas_bronze,soma_total_removida,pct_da_receita_bronze
0,data impossível (antes do período),15,0.001,352.31,0.000
1,total_amount negativo,35500,1.197,-893347.54,-1.124
2,total_amount igual a zero,413,0.014,0.00,0.000
3,tarifa negativa,2064,0.070,5260.95,0.007
4,distância negativa,0,0.000,0.00,0.000
5,total_amount acima do limite,7,0.000,21342.80,0.027
7,escopo: embarque fora de jan/2024 (depois do p...,3,0.000,90.47,0.000



<class 'pandas.DataFrame'>
RangeIndex: 2926603 entries, 0 to 2926602
Data columns (total 11 columns):
 #   Column            Dtype         
---  ------            -----         
 0   data_embarque     datetime64[us]
 1   id_zona_embarque  int32         
 2   valor_tarifa      float64       
 3   valor_total       float64       
 4   distancia_milhas  float64       
 5   _data_extracao    str           
 6   _origem           str           
 7   bairro            str           
 8   zona              str           
 9   zona_servico      str           
 10  hora_embarque     int8          
dtypes: datetime64[us](1), float64(3), int32(1), int8(1), str(5)
memory usage: 588.5 MB


,data_embarque,id_zona_embarque,valor_tarifa,valor_total,distancia_milhas,_data_extracao,_origem,bairro,zona,zona_servico,hora_embarque
0,2024-01-01 00:57:55,186,17.7,22.70,1.72,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/trip-dat...,Manhattan,Penn Station/Madison Sq West,Yellow Zone,0
1,2024-01-01 00:03:00,140,10.0,18.75,1.80,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/trip-dat...,Manhattan,Lenox Hill East,Yellow Zone,0
2,2024-01-01 00:17:06,236,23.3,31.30,4.70,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/trip-dat...,Manhattan,Upper East Side North,Yellow Zone,0
3,2024-01-01 00:36:38,79,10.0,17.00,1.40,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/trip-dat...,Manhattan,East Village,Yellow Zone,0
4,2024-01-01 00:46:51,211,7.9,16.10,0.80,2026-10-22T00:29:39,https://d37ci6vzurychx.cloudfront.net/trip-dat...,Manhattan,SoHo,Yellow Zone,0


Nulos por coluna na Silver:


,data_embarque,id_zona_embarque,valor_tarifa,valor_total,distancia_milhas,_data_extracao,_origem,bairro,zona,zona_servico,hora_embarque
nulos,0,0,0,0,0,0,0,0,0,0,0


Período da Silver: 2024-01-01 00:00:00 -> 2024-01-31 23:59:55


### Decisões de limpeza da Silver (formato: decidimos X porque Y)

**Padronização.** Decidimos renomear as colunas para português em `snake_case` (`data_embarque`, `id_zona_embarque`, `valor_tarifa`, `valor_total`, `distancia_milhas`) e converter datas para `datetime` e valores para numérico, porque nomes e tipos consistentes evitam erro nas camadas seguintes. A conversão não criou nenhum nulo (0).

**Duplicados.** Decidimos remover linhas idênticas nas 5 colunas de dado (a Bronze só guarda 5 colunas), ignorando os metadados na comparação, porque duas linhas iguais no mesmo segundo, na mesma zona e com o mesmo valor e distância são quase certamente registro repetido. Foram removidos **19** registros (de 2.964.624 para 2.964.605). Risco assumido: como a Bronze não tem a hora de desembarque, não dá para distinguir uma corrida legítima idêntica; com 19 casos em 2,9 milhões, o impacto é desprezível.

**Nulos em viagens.** Decidimos remover linhas com nulo nas 5 colunas usadas, porque são essenciais para a pergunta. Não houve nenhuma (0 removidas). Os nulos de `passenger_count`, `RatecodeID` e taxas (140.162 linhas, 4,73%) ficaram fora desde o Extract, porque a pergunta não usa essas colunas.

**Nulos em zonas.** Decidimos preencher os textos `N/A` com rótulos explícitos (`Fora de NYC`, `Zona desconhecida`, `Sem classificação`) em vez de remover ou deixar como nulo, porque eles significam "zona não identificada" (LocationID 264 e 265) e as corridas dessas zonas são receita real. Foram rotulados 1 `bairro`, 1 `zona` e 2 `zona_servico`.

**total_amount negativo ou zero.** Decidimos remover, porque receita negativa ou zero não representa uma corrida faturada (estornos, disputas, corridas canceladas) e distorceria a soma por zona e hora. Afetaram **35.500** e **413** registros. Na Bronze havia 35.504 e 416: a diferença são registros que já tinham saído como duplicados ou data impossível.

**Tarifa e distância negativas.** Decidimos remover, porque são valores impossíveis para uma corrida. A tarifa negativa (com total positivo) removeu **2.064** registros; a distância negativa, **0**.

**Valor acima do limite.** Decidimos tratar como inválido `total_amount` acima de US$ 1.000 (parâmetro `total_max`), porque corridas assim em um mês de táxi amarelo são quase certamente erro de registro e um único valor extremo (o máximo da Bronze é US$ 5.000) poderia distorcer o ranking. Removeu **7** registros, somando US$ 21.342,80.

**Distância zero.** Decidimos manter, porque a pergunta é sobre receita e essas corridas (60.371 na Bronze, 56.649 na Silver) têm valor cobrado; `trip_distance` só importa na segunda pergunta (Bloco 9).

**Datas.** Decidimos remover embarques anteriores a 01/01/2024 (data impossível, como 2002) e, como filtro de escopo, os de 01/02/2024 em diante, porque a pergunta é sobre janeiro de 2024. As duas regras descartaram **15 + 3 = 18** registros, os mesmos 18 vistos no Bloco 2.

**Junção.** Decidimos fazer `left join` com `validate="many_to_one"` pela chave `PULocationID = LocationID`, porque a tabela de zonas tem uma linha por zona e não pode multiplicar viagens.

# Bloco 05 |

# Bloco 06 |

# Bloco 07 |

# Bloco 08 |

# Bloco 09 |

# Bloco 10 |

# Bloco 11 |